In [1]:
import sys
import os
import glob

print("--- DIAGNOSTYKA I NAPRAWA ---")

# 1. Ustawiamy ścieżki (potwierdzone Twoim screenem)
spark_home = "/usr/local/spark"
os.environ["JAVA_HOME"] = "/usr/local/java"
os.environ["SPARK_HOME"] = spark_home

# 2. Szukamy pliku py4j (serce komunikacji z Javą)
# W Spark 3.1.2 powinien to być py4j-0.10.9-src.zip
py4j_path = glob.glob(os.path.join(spark_home, 'python', 'lib', 'py4j-*-src.zip'))

if not py4j_path:
    print("BŁĄD KRYTYCZNY: Nie widzę pliku py4j w folderze:", os.path.join(spark_home, 'python', 'lib'))
    # Sprawdzamy co tam w ogóle jest
    print("Zawartość folderu lib:", os.listdir(os.path.join(spark_home, 'python', 'lib')))
else:
    print(f"Znaleziono sterownik Javy: {py4j_path[0]}")
    
    # 3. Dodajemy te pliki do Pythona "na siłę" (przed wszystkim innym)
    sys.path.insert(0, py4j_path[0])                 # Dodaj zipa
    sys.path.insert(0, os.path.join(spark_home, 'python')) # Dodaj folder python
    
    # 4. Dopiero teraz importujemy Sparka
    try:
        from pyspark.sql import SparkSession
        
        spark = SparkSession.builder \
            .appName("FinnhubRatunek") \
            .master("local[*]") \
            .getOrCreate()
            
        print("\nSUKCES! Spark działa.")
        print("Wersja Sparka:", spark.version)
        print("Katalog domowy Sparka:", os.environ["SPARK_HOME"])
        
    except Exception as e:
        print("\nNadal błąd przy uruchamianiu sesji:")
        print(e)

--- DIAGNOSTYKA I NAPRAWA ---
Znaleziono sterownik Javy: /usr/local/spark/python/lib/py4j-0.10.9-src.zip


26/01/09 23:26:43 WARN util.Utils: Your hostname, node1 resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/01/09 23:26:43 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/01/09 23:26:44 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 23:26:47 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.



SUKCES! Spark działa.
Wersja Sparka: 3.1.2
Katalog domowy Sparka: /usr/local/spark


In [2]:
from pyspark.sql.functions import col, explode

# Spark automatycznie wykryje partycje dt=... i hr=...
# Nie musisz podawać konkretnej daty, podaj folder główny!
df_raw = spark.read.json("/user/vagrant/finnhub_trades")

# Zobaczmy co wykrył
df_raw.printSchema()

26/01/09 23:27:00 WARN datasources.SharedInMemoryCache: Evicting cached table partition metadata from memory due to size constraints (spark.sql.hive.filesourcePartitionFileCacheSize = 262144000 bytes). This may impact query planning performance.
                                                                                

root
 |-- conditions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- raw: struct (nullable = true)
 |    |-- c: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- p: double (nullable = true)
 |    |-- s: string (nullable = true)
 |    |-- t: long (nullable = true)
 |    |-- v: double (nullable = true)
 |-- received_at_ms: long (nullable = true)
 |-- rowkey: string (nullable = true)
 |-- source: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- trade_ts_ms: long (nullable = true)
 |-- volume: double (nullable = true)
 |-- dt: date (nullable = true)
 |-- hr: integer (nullable = true)



In [3]:
from pyspark.sql.functions import from_unixtime, col

# Wybieramy tylko konkrety + zamieniamy czas z milisekund na czytelną datę
df_trades = df_raw.select(
    col("symbol"),
    col("price"),
    col("volume"),
    col("dt"),  # data z nazwy folderu
    col("hr"),  # godzina z nazwy folderu
    from_unixtime(col("trade_ts_ms") / 1000).alias("transaction_time") # czytelny czas
)

# Zobaczmy jak to teraz wygląda (powinno być ładnie i czytelnie)
df_trades.show(5)

+---------------+--------+-------+----------+---+-------------------+
|         symbol|   price| volume|        dt| hr|   transaction_time|
+---------------+--------+-------+----------+---+-------------------+
|BINANCE:ETHUSDT| 3111.45| 0.0026|2026-01-09| 15|2026-01-09 15:54:05|
|BINANCE:BTCUSDT|91185.69|0.00498|2026-01-09| 15|2026-01-09 15:54:05|
|BINANCE:BTCUSDT|91185.69|0.00471|2026-01-09| 15|2026-01-09 15:54:05|
|BINANCE:BTCUSDT|91185.69| 6.0E-5|2026-01-09| 15|2026-01-09 15:54:05|
|BINANCE:BTCUSDT|91185.51| 6.0E-5|2026-01-09| 15|2026-01-09 15:54:05|
+---------------+--------+-------+----------+---+-------------------+
only showing top 5 rows



In [4]:
from pyspark.sql.functions import col, desc

# 1. Grupujemy po symbolu i liczymy wystąpienia
# (Możesz użyć df_raw albo df_trades - zależy co masz teraz wczytane)
symbol_counts = df_raw.groupBy("symbol").count()

# 2. Sortujemy wynik, żeby na górze były te najczęstsze
# (col("count") to domyślna nazwa kolumny, którą Spark tworzy po zliczeniu)
result = symbol_counts.orderBy(col("count").desc())

# 3. Wyświetlamy wynik
result.show()

+---------------+-------+
|         symbol|  count|
+---------------+-------+
|BINANCE:BTCUSDT|1461091|
|BINANCE:ETHUSDT| 725768|
|           AAPL|  11011|
|           INTC|   7289|
|   IC MARKETS:1|   6647|
|           AMZN|   6095|
|           NVDA|   3987|
|            AMD|    598|
|          BRK.B|    345|
+---------------+-------+



In [5]:
from pyspark.sql.functions import col

# 1. Filtrujemy tylko wiersze, gdzie symbol to "BRK.B"
df_berkshire = df_trades.filter(col("symbol") == "BRK.B")

# 2. Wyświetlamy wynik
print("--- Transakcje dla BRK.B ---")
df_berkshire.show(10, truncate=False)

# 3. (Opcjonalnie) Sprawdźmy ile ich w ogóle złapało
count = df_berkshire.count()
print(f"Liczba znalezionych transakcji BRK.B: {count}")

--- Transakcje dla BRK.B ---


+------+------+------+----------+---+-------------------+
|symbol|price |volume|dt        |hr |transaction_time   |
+------+------+------+----------+---+-------------------+
|BRK.B |498.41|40.0  |2026-01-09|17 |2026-01-09 17:46:15|
|BRK.B |498.41|40.0  |2026-01-09|17 |2026-01-09 17:46:15|
|BRK.B |498.37|40.0  |2026-01-09|17 |2026-01-09 17:46:15|
|BRK.B |498.39|40.0  |2026-01-09|17 |2026-01-09 17:46:15|
|BRK.B |498.39|40.0  |2026-01-09|17 |2026-01-09 17:45:37|
|BRK.B |497.9 |43.0  |2026-01-09|17 |2026-01-09 17:16:27|
|BRK.B |497.9 |50.0  |2026-01-09|17 |2026-01-09 17:16:27|
|BRK.B |497.89|43.0  |2026-01-09|17 |2026-01-09 17:16:27|
|BRK.B |497.63|90.0  |2026-01-09|16 |2026-01-09 16:58:28|
|BRK.B |497.59|80.0  |2026-01-09|16 |2026-01-09 16:58:28|
+------+------+------+----------+---+-------------------+
only showing top 10 rows



[Stage 11:===================================================>(2622 + 2) / 2625]

Liczba znalezionych transakcji BRK.B: 345


In [6]:
from pyspark.sql.functions import col

# 1. Wczytanie danych z Parquet
# (Podaj ścieżkę do folderu, w którym zapisałeś pliki parquet)
df_parquet = spark.read.parquet("/user/vagrant/finnhub_parquet")

# 2. Policzenie wszystkich wierszy w tabeli
total_rows = df_parquet.count()
print(f"Łączna liczba wierszy: {total_rows}")

# 3. Podliczenie ile jest wartości i jakich w kolumnie 'symbol'
# Grupujemy po symbolu, liczymy i sortujemy malejąco (żeby widzieć te najczęstsze)
print("--- Statystyka dla kolumny SYMBOL ---")
df_parquet.groupBy("symbol") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

Łączna liczba wierszy: 2222829
--- Statystyka dla kolumny SYMBOL ---


[Stage 17:==============================================>       (171 + 3) / 200]

+---------------+-------+
|         symbol|  count|
+---------------+-------+
|BINANCE:BTCUSDT|1461091|
|BINANCE:ETHUSDT| 725766|
|           AAPL|  11011|
|           INTC|   7289|
|   IC MARKETS:1|   6647|
|           AMZN|   6095|
|           NVDA|   3987|
|            AMD|    598|
|          BRK.B|    345|
+---------------+-------+



In [7]:
df_parquet.printSchema()

root
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- trade_ts_ms: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- source: string (nullable = true)
 |-- received_at_ms: long (nullable = true)



In [8]:
df_parquet.show(8)

+---------------+--------+-------+-------------+----------+-------+--------------+
|         symbol|   price| volume|  trade_ts_ms|event_type| source|received_at_ms|
+---------------+--------+-------+-------------+----------+-------+--------------+
|BINANCE:BTCUSDT| 91185.7| 6.0E-5|1767974045464|     trade|finnhub| 1767974046026|
|BINANCE:BTCUSDT|91185.69|0.00219|1767974045464|     trade|finnhub| 1767974046027|
|BINANCE:ETHUSDT| 3111.45| 0.0017|1767974045467|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT|91185.69|0.00127|1767974045469|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT|91185.69| 2.2E-4|1767974045469|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT|91185.69| 6.0E-5|1767974045469|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT| 91185.5| 6.0E-5|1767974045469|     trade|finnhub| 1767974046028|
|BINANCE:ETHUSDT|  3110.9|  0.009|1767974045545|     trade|finnhub| 1767974046028|
+---------------+--------+-------+-------------+----------+-------+--------------+
only